# Time series extraction
---

Running this notebook you will:

- Compute and verify the neural-muscular ratio of the organoids using the brightfield images
- Extract timeseries for each video recording of an area of the organoid representing the contraction at the organoid border
- Save the timeseries and update your working spreadsheet with the computed ratios along with other features related to this analysis

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import time
import threading

%matplotlib inline
%matplotlib widget
from matplotlib import pyplot as plt
import ipywidgets as ipyw

from readlif.reader import LifFile
from skimage.io import imread, imsave

import sys
sys.path.append('../utils/')
from time_series_extraction import *
from imaging_viewers import *

import warnings
warnings.filterwarnings("ignore")

## Step 1.A. Data path for brightfield images (N-M ratio)
- Give the path to your data (images):
- Give the path of the folder where to save the masks:

In [2]:
# add input images dir path widget
src_filepath_imgs_wig = ipyw.widgets.Text(
    value='images',
    placeholder='Insert path here',
    description='Source images filepath:',
    disabled=True,
    style= {'description_width': 'initial'},
    layout=ipyw.widgets.Layout(width='95%')
)
checkbox_src_filepath_imgs = ipyw.widgets.Checkbox(
    value=False,
    description='edit path',
    disabled=False,
    indent=False
)

def src_filepath_change_imgs(c):
    if checkbox_src_filepath_imgs.value:
        src_filepath_imgs_wig.disabled = False
    else:
        src_filepath_imgs_wig.disabled = True

checkbox_src_filepath_imgs.observe(src_filepath_change_imgs, names="value")

# add masks dest dir path widget
dst_filepath_masks_wig = ipyw.widgets.Text(
    value='masks',
    placeholder='Insert path here',
    description='Destination masks filepath:',
    disabled=True,
    style= {'description_width': 'initial'},
    layout=ipyw.widgets.Layout(width='95%')
)
checkbox_dst_filepath_masks = ipyw.widgets.Checkbox(
    value=False,
    description='edit path',
    disabled=False,
    indent=False
)

def dst_filepath_change_masks(c):
    if checkbox_dst_filepath_masks.value:
        dst_filepath_masks_wig.disabled = False
    else:
        dst_filepath_masks_wig.disabled = True

checkbox_dst_filepath_masks.observe(dst_filepath_change_masks, names="value")

In [3]:
# display widgets
display(ipyw.widgets.HBox([src_filepath_imgs_wig, checkbox_src_filepath_imgs]))
display(ipyw.widgets.HBox([dst_filepath_masks_wig, checkbox_dst_filepath_masks]))

## Step 1.B. Data path for video data
- Give the path to your data (lif video files)
- Give the path of the folder where to save the extracted signals

In [4]:
# add input videos dir path widget
src_filepath_vids_wig = ipyw.widgets.Text(
    value='videos',
    placeholder='Insert path here',
    description='Source videos filepath:',
    disabled=True,
    style= {'description_width': 'initial'},
    layout=ipyw.widgets.Layout(width='95%')
)
checkbox_src_vids_filepath = ipyw.widgets.Checkbox(
    value=False,
    description='edit path',
    disabled=False,
    indent=False
)

def src_filepath_change_vids(c):
    if checkbox_src_vids_filepath.value:
        src_filepath_vids_wig.disabled = False
    else:
        src_filepath_vids_wig.disabled = True

checkbox_src_vids_filepath.observe(src_filepath_change_vids, names="value")

# add signals dest dir path widget
dst_filepath_sigs_wig = ipyw.widgets.Text(
    value='extracted_signals',
    placeholder='Insert path here',
    description='Destination time signals filepath:',
    disabled=True,
    style= {'description_width': 'initial'},
    layout=ipyw.widgets.Layout(width='95%')
)
checkbox_dst_sigs_filepath = ipyw.widgets.Checkbox(
    value=False,
    description='edit path',
    disabled=False,
    indent=False
)

def dst_filepath_change_sigs(c):
    if checkbox_dst_sigs_filepath.value:
        dst_filepath_sigs_wig.disabled = False
    else:
        dst_filepath_sigs_wig.disabled = True

checkbox_dst_sigs_filepath.observe(dst_filepath_change_sigs, names="value")

In [5]:
# display widgets
display(ipyw.widgets.HBox([src_filepath_vids_wig, checkbox_src_vids_filepath]))
display(ipyw.widgets.HBox([dst_filepath_sigs_wig, checkbox_dst_sigs_filepath]))

## Step 1.C. Select working datasheet
If you choose to work on a different file from that displayed **then** select it from the dropdown menu (it has to be in the same location as this notebook).

In [6]:
# get list of available spreadsheet in current directory
files = [file for file in os.listdir('.') if (file.endswith('.xlsx') or file.endswith('.csv') or file.endswith('.xls'))]

ds_wig = ipyw.Dropdown(
    options=files,
    description=' Working Datasheet:',
    style= {'description_width': 'initial'},
    layout=ipyw.widgets.Layout(width='95%')
)
display(ds_wig)

# create output widget to print selected spreasheet
output_wig = ipyw.widgets.Output()

# clear the output at every execution
@output_wig.capture(clear_output=True)
def print_output(b=None):
    with output_wig: print('Going to write to excel sheet: ', ds_wig.value)

print_output()
ds_wig.observe(print_output, names="value")

# display output
display(output_wig)



Dropdown(description=' Working Datasheet:', layout=Layout(width='95%'), options=('Datasheet_template_example_p…

Output()

## Step 2.A. Image file selection
If you choose to work on a different image from that displayed **then** select it from the dropdown menu.

In [7]:
if os.path.exists(src_filepath_imgs_wig.value):
    files = [file for file in os.listdir(src_filepath_imgs_wig.value) if file.endswith('.tif')]
else:
    files = []

wig_img = ipyw.Dropdown(
    options=files,
    value=files[0] if files else None,
    description='image:',
    layout={'width': '50%'},
    #disabled=True
)

def src_filepath_imgs_change(b=None):
    if os.path.exists(src_filepath_imgs_wig.value):
        wig_img.options = [file for file in os.listdir(src_filepath_imgs_wig.value) if file.endswith('.tif')]
        wig_img.value = wig_img.options[0]
    else:
        wig_img.options = []
        wig_img.value = None

src_filepath_imgs_wig.observe(src_filepath_imgs_change, names="value")

display(wig_img)

Dropdown(description='image:', layout=Layout(width='50%'), options=('P1365_SMA_BRAN_d60_O4.tif', 'P1365_SMA_BR…

## Step 2.B. N-M ratio computation
Click the button below to compute the neural muscular ratio.

In [8]:
# When the button is clicked, read the image selected by the user, compute thresholds,
# get mask and n-m ratios and display results
out_viewer = ipyw.widgets.Output()

@out_viewer.capture(clear_output=True)
def compute_ratios(b=None):
    compute_nm_ratio_btn.disabled = True
    img_filename =  wig_img.value
    global img # not sure if this is good practise?
    img = imread(os.path.join(src_filepath_imgs_wig.value, img_filename), as_gray=True)
    global viewer
    viewer = Viewer(img)
    compute_nm_ratio_btn.disabled = False
    apply_post_btn.disabled = False
    
compute_nm_ratio_btn = ipyw.widgets.Button(
    description='Compute N-M ratio',
    disabled=False,
    button_style='',
    layout=ipyw.widgets.Layout(width='30%')
)

compute_nm_ratio_btn.on_click(compute_ratios)

display(compute_nm_ratio_btn)
display(out_viewer)

Button(description='Compute N-M ratio', layout=Layout(width='30%'), style=ButtonStyle())

Output()

## Step 2.C. Check post processing
Click the button below to see the result of applying post-processing on the mask of the previous step. Two post-processing methods are applied:
* Small holes are filled
* Small objects are removed

In [9]:
from matplotlib import pyplot as plt
from matplotlib.widgets import Slider, Button, RadioButtons

class ViewerPP():
    
    def __init__(self, img, mask, n_ratio, m_ratio, total_area):

        self.img = img
        self.img_array = np.ma.masked_array(self.img, ~np.zeros(self.img.shape).astype(bool))
        self.masked_array = np.ma.masked_array(mask, ~mask.astype(bool))
          
        fig_title = "Mask after postprocessing\n Muscle part: "+str(m_ratio)+", Neural part: "+str(n_ratio)+',\n Total_area: '+str(round(total_area, 3))+' mm^2'
        # Display image and overlay the mask
        self.fig, self.ax = plt.subplots()
        self.ax.imshow(self.img, cmap='gray')
        self.ax.set_axis_off()
        self.ax.set_title(fig_title)
          
        # Add a checkbox to switch between views
        radio_ax = self.fig.add_axes([0.8, 0.8, 0.18, 0.2])
        self.radio_btn = RadioButtons(radio_ax, ('mask on', 'mask off'))      
        self.radio_btn.on_clicked(self.change_view)
        # Overlay the mask on the image
        self.masked_image = self.ax.imshow(self.masked_array, cmap='viridis')
        
        plt.show()
    
    # Switch between simple image view and overlay with mask view
    def change_view(self, label):
        if label=='mask on':
            self.masked_image.set_array(self.masked_array)
        else:
            self.masked_image.set_array(self.img_array)
        self.fig.canvas.draw_idle()
    
      

In [10]:
# When the button is clicked, read the image selected by the user, compute thresholds,
# get mask and n-m ratios and display results
out_post = ipyw.widgets.Output()

@out_post.capture(clear_output=True)
def apply_postprocessing(b=None):
    apply_post_btn.disabled = True
    
    global n_ratio, m_ratio
    n_ratio, m_ratio = viewer.get_ratios()
    global mask
    mask = viewer.get_mask()
    global total_area
    total_area = compute_total_area(mask)
    global mask_pp
    mask_pp = remove_holes_and_objects(mask.copy(), min_size=2000)
    global n_ratio_pp, m_ratio_pp
    n_ratio_pp, m_ratio_pp = compute_nm_ratio(mask_pp)
    global total_area_pp
    total_area_pp = compute_total_area(mask_pp)
    global viewer_pp
    viewer_pp = ViewerPP(img, mask_pp, n_ratio_pp, m_ratio_pp, total_area_pp)
    
    apply_post_btn.disabled = False
    save_res_btn.disabled = False
    
apply_post_btn = ipyw.widgets.Button(
    description='Apply post-processing',
    disabled=True,
    button_style='',
    layout=ipyw.widgets.Layout(width='30%')
)
apply_post_btn.on_click(apply_postprocessing)

display(apply_post_btn)
display(out_post)

Button(description='Apply post-processing', disabled=True, layout=Layout(width='30%'), style=ButtonStyle())

Output()

## Step 2.D. Include Post-processing?
Only click the box below if the mask above seems better than in the previous step.


In [11]:
post_processing_wig = ipyw.widgets.Checkbox(
    value=False,
    description='Postprocessing (Y/N)',
    disabled=False,
    indent=False
)

def post_processing_change(c):
    pass

post_processing_wig.observe(post_processing_change, names="value")

display(post_processing_wig)

Checkbox(value=False, description='Postprocessing (Y/N)', indent=False)

## Step 2.E.  Write NMO ratios to datasheet

In [12]:
out_res = ipyw.widgets.Output()

@out_res.capture(clear_output=True)
def save_results(b=None):
    save_res_btn.disabled = True
    global n_ratio, m_ratio, total_area, mask
    # check if post-processing was selected and if so update n-m ratios and mask
    if post_processing_wig.value==True:
        n_ratio = n_ratio_pp
        m_ratio = m_ratio_pp
        mask = mask_pp
        total_area = total_area_pp

    muscle_thresh, neural_thresh = viewer.get_thresholds()
    
    print('Opening datasheet: ', ds_wig.value)
    data_info = pd.read_excel(ds_wig.value)
    df_columns = data_info.columns
    # rename column if needed
    if 'File_video name' in df_columns: data_info.rename(columns={'File_video name': 'Video name'}, inplace=True) # TO DO: remove in new version if we use new template
    
    # If columns don't exist create them
    if 'Image name' not in df_columns: data_info['Image name'] =  pd.Series(dtype='str')
    if 'Neural Ratio' not in df_columns: data_info['Neural Ratio'] =  pd.Series(dtype='float')
    if 'Muscle Ratio' not in df_columns: data_info['Muscle Ratio'] =  pd.Series(dtype='float')
    if 'Threshold Neural' not in df_columns: data_info['Threshold Neural'] =  pd.Series(dtype='float')
    if 'Threshold Muscle' not in df_columns: data_info['Threshold Muscle'] =  pd.Series(dtype='float')
    if 'Post Processing NM Ratio' not in df_columns: data_info['Post Processing NM Ratio'] = pd.Series(dtype='bool')
    if 'Total Area [mm^2]' not in df_columns: data_info['Total Area [mm^2]'] = pd.Series(dtype='int')
    
    # Add chosen threshold value to the info file
    img_filename =  wig_img.value
    found_ids = data_info.index[data_info['Video name'].str.contains(Path(img_filename).stem) == True].tolist()
    if len(found_ids)==0:
        print("Filename ", img_filename, "not found in datasheet. Creating a new entry. Please fill in the rest of the columns manually.")
        # Define a new row
        new_row = {'Image name': Path(img_filename).stem,
                   'Neural Ratio': n_ratio,
                   'Muscle Ratio': m_ratio,
                   'Threshold Neural': neural_thresh,
                   'Threshold Muscle': muscle_thresh,
                   'Post Processing NM Ratio': post_processing_wig.value,
                   'Total Area [mm^2]': total_area
                  }
        # Append the new row to the dataframe
        data_info = pd.concat([data_info, pd.DataFrame([new_row])], ignore_index=True)

      
    else:
        if len(found_ids)>1:
            print("More than one entries in data sheet found corresponding to the image ", img_filename)
        for id in found_ids:
            print("Adding neural and muscle ratios to ", data_info.loc[id, 'Video name'])
            data_info.loc[id, 'Image name'] = Path(img_filename).stem
            data_info.loc[id, 'Neural Ratio'] = n_ratio
            data_info.loc[id, 'Muscle Ratio'] = m_ratio
            data_info.loc[id, 'Threshold Neural'] = neural_thresh
            data_info.loc[id, 'Threshold Muscle'] = muscle_thresh
            data_info.loc[id, 'Post Processing NM Ratio'] = post_processing_wig.value
            data_info.loc[id, 'Total Area [mm^2]'] = total_area


    # And save the result
    data_info.to_excel(ds_wig.value, index=False)

    dst_filepath_masks = dst_filepath_masks_wig.value
    if not os.path.exists(dst_filepath_masks): os.mkdir(dst_filepath_masks)
    imsave(os.path.join(dst_filepath_masks, img_filename), mask)
    save_res_btn.disabled = False

    
save_res_btn = ipyw.widgets.Button(
    description='Save results',
    disabled=True,
    button_style='',
    layout=ipyw.widgets.Layout(width='30%')
)
save_res_btn.on_click(save_results)

display(save_res_btn)
display(out_res)

Button(description='Save results', disabled=True, layout=Layout(width='30%'), style=ButtonStyle())

Output()

## Step 3. Video File selection
- If you choose to work on a different video from that displayed **then** select it from the dropdown menu. 
- You can also select the video channel you want to work with in case of multi-channel inputs. If you have only one channel you can leave it to the default value 0.

In [13]:
if os.path.exists(src_filepath_vids_wig.value):
    files = [file for file in os.listdir(src_filepath_vids_wig.value) if file.endswith('.lif')]
else:
    files = []

wig_vid = ipyw.Dropdown(
    options=files,
    value=files[0] if files else None,
    description='video:',
    layout=ipyw.widgets.Layout(width='95%')
)

def src_filepath_vids_change(b=None):
    if os.path.exists(src_filepath_vids_wig.value):
        wig_vid.options = [file for file in os.listdir(src_filepath_vids_wig.value) if file.endswith('.lif')]
        wig_vid.value = wig_vid.options[0]
    else:
        wig_vid.options = []
        wig_vid.value = None

src_filepath_vids_wig.observe(src_filepath_vids_change, names="value")


# load video channel
channel_wig = ipyw.widgets.Text(
    value='0',
    placeholder='0',
    description='channel: ',
    disabled=True,
    style= {'description_width': 'initial'},
    layout=ipyw.widgets.Layout(width='20%')
)

checkbox_channel = ipyw.widgets.Checkbox(
    value=False,
    description='edit channel',
    disabled=False,
    indent=False
)

def channel_edit(c):
    if checkbox_channel.value:
        channel_wig.disabled = False
    else:
        channel_wig.disabled = True

checkbox_channel.observe(channel_edit, names="value")

# display widgets
display(ipyw.widgets.HBox([wig_vid, channel_wig, checkbox_channel]))


## Step 4. Orgnanoid border detection
Click the button below to run the threshold detector which will result in finding the border of the organoid.


In [14]:
progress_bar= ipyw.widgets.FloatProgress(
    value=0,
    min=0,
    max=1,
    description='',
    bar_style=''
)

def update_progress_bar(b=None):
    global computed
    increments = 0.25
    while True:
        if computed:
            progress_bar.value = 1
            progress_bar.description = 'done'
            progress_bar.bar_style = 'success'
            return
        else:
            if progress_bar.value > (1 - increments):
                progress_bar.value = 0
            else:
                progress_bar.value = progress_bar.value + increments
            progress_bar.description = 'computing'
            progress_bar.bar_style = 'warning'
        time.sleep(.5)

In [15]:
out_thresh = ipyw.widgets.Output()

computed = False
@out_thresh.capture(clear_output=True)
def compute_thresh(b=None):
    run_thresh_btn.disabled = True
    global computed
    computed = False
    progress_bar_thread = threading.Thread(target=update_progress_bar)
    progress_bar_thread.start()
    
    files = [file for file in os.listdir(src_filepath_vids_wig.value) if file.endswith('.lif')]
    file_id = files.index(wig_vid.value)
    # read image
    img = LifFile(os.path.join(src_filepath_vids_wig.value, files[file_id]))
    img = img.get_image(0)
    # get channel
    c = int(channel_wig.value)
    with out_thresh:
        print("Opening file: ", files[file_id])
        print(f'You selected channel {c} in the previous step')
    global img_array
    img_array = [i for i in img.get_iter_t(c=1, z=0)] #c=0 unique channel, c=1 for second channel
    img_array = np.stack(img_array)
    # compute threshold
    global img_gaussian
    global thresh
    thresh, min_int, max_int = get_thresh(img_array[0])
    img_gaussian = compute_gaussian(img_array, 2)
    thresh = round(thresh,2)
    print('Default threshold value is: ', thresh)
    run_thresh_btn.disabled = False
    computed = True
    progress_bar_thread.join()
    view_thresh_btn.disabled = False
    
run_thresh_btn = ipyw.widgets.Button(
    description='Compute threshold',
    disabled=False,
    button_style='',
    layout=ipyw.widgets.Layout(width='30%')
)
run_thresh_btn.on_click(compute_thresh)

display(run_thresh_btn)
display(progress_bar)
display(out_thresh)

Button(description='Compute threshold', layout=Layout(width='30%'), style=ButtonStyle())

FloatProgress(value=0.0, max=1.0)

Output()

## Step 5. Verifying threshold
Click the button below to have a look at the border we get with the automatic threshold.
You can scroll through the video frames and change the default threshold to choose a value which will result in a good organoid/background seperation. The higher the threhsold the more lighter regions will be included in the foreground.

**Note:** This may take some time to load - please be patient!

In [16]:
out_verify = ipyw.widgets.Output()

computed = False
@out_verify.capture(clear_output=True)
def verify_thresh(b=None):
    global computed
    view_thresh_btn.disabled = True

    computed = False
    progress_bar_thread = threading.Thread(target=update_progress_bar_verify_thr)
    progress_bar_thread.start()
    ImageSliceViewer3D(img_gaussian, img_array, thresh)
    view_thresh_btn.disabled = False
    computed = True
    progress_bar_thread.join()
    save_extract_btn.disabled = False
    
view_thresh_btn = ipyw.widgets.Button(
    description='Verify threshold',
    disabled=True,
    button_style='',
    layout=ipyw.widgets.Layout(width='30%')
)
view_thresh_btn.on_click(verify_thresh)


In [17]:
progress_bar_verify_thr = ipyw.widgets.FloatProgress(
    value=0,
    min=0,
    max=1,
    description='',
    bar_style=''
)

def update_progress_bar_verify_thr(b=None):
    global computed
    increments = 0.25
    while True:
        if computed:
            progress_bar_verify_thr.value = 1
            progress_bar_verify_thr.description = 'done'
            progress_bar_verify_thr.bar_style = 'success'
            return
        else:
            if progress_bar_verify_thr.value > (1 - increments):
                progress_bar_verify_thr.value = 0
            else:
                progress_bar_verify_thr.value = progress_bar_verify_thr.value + increments
            progress_bar_verify_thr.description = 'computing'
            progress_bar_verify_thr.bar_style = 'warning'
        time.sleep(.5)


In [18]:
display(view_thresh_btn)
display(progress_bar_verify_thr)
display(out_verify)

Button(description='Verify threshold', disabled=True, layout=Layout(width='30%'), style=ButtonStyle())

FloatProgress(value=0.0, max=1.0)

Output()

## Step 6. Final threhsold confirmation and signal extraction
Set the threshold you chose in the previous step and then click the button **"Extract & Save"** signals. The following two steps will be performed:
1. The threshold will be added to your spreadsheet.
2. Then selected threshold will be used to extract and save the border signals which can be used for the analysis step.

In [19]:
out_write_thresh = ipyw.widgets.Output()

# add input videos dir path widget
thresh_wig = ipyw.widgets.BoundedFloatText(
    value=None,
    min=0.,
    max=1.,
    step=.01,
    description='Border threshold:',
    style= {'description_width': 'initial'},
    disabled=False
)

@out_write_thresh.capture(clear_output=True)
def save_and_extract(b):
    save_extract_btn.disabled = True
    # get final threshold
    thresh = float(thresh_wig.value)
    # Open data info file
    datafile = ds_wig.value
    print('Opening datasheet: ', datafile)
    data_info = pd.read_excel(datafile)
    df_columns = data_info.columns
    
    # rename column if needed
    if 'File_video name' in df_columns: data_info.rename(columns={'File_video name': 'Video name'}, inplace=True)
    data_info['Video name'] = data_info['Video name'].apply(lambda x: os.path.splitext(os.path.basename(str(x)))[0])
    
    # Add chosen threshold value to the info file
    # data_info.insert(10,'Threshold', 'default') # only first time
    data_info.loc[data_info['Video name'] == os.path.splitext(os.path.basename(wig_vid.value))[0], 'Threshold organoid border'] = thresh
    data_info.to_excel(datafile, index=False)

    # signal extraction
    bar = ipyw.IntProgress(value=0, min=0, max=2*img_array.shape[0]+1, description='Extracting signal...', bar_style='',)
    display(bar)
    global subsample_regions
    global distances
    distances, subsample_regions = extract_signal(img_array, thresh, bar) # this takes a long time!!
    # save result in numpy file
    for i in range(subsample_regions):
      distances[i, :] -= distances[i,0]
    bar.bar_style = 'success'
    # get current working file
    files = [file for file in os.listdir(src_filepath_vids_wig.value) if file.endswith('.lif')]
    file_id = files.index(wig_vid.value)
    new_file = os.path.join(dst_filepath_sigs_wig.value, Path(files[file_id]).stem+'.npy')
    np.save(new_file, distances) #add dst_path
    
    print(subsample_regions, ' signals were extracted from this video file and saved in: ', new_file)
    
    d = np.load(new_file)
    if np.any(np.isnan(d)):
     print('There may be a problem with this file, as it includes NaN values which can cause problems in the downstream analysis. Please check again that your threshold makes sense. If the problem persists please contact the Helmholtz AI consultants.')
    save_extract_btn.disabled = False
    plot_btn.disabled = False
    
save_extract_btn = ipyw.widgets.Button(
    description='Extract & Save',
    disabled=True,
    button_style='',
    layout=ipyw.widgets.Layout(width='20%')
)
save_extract_btn.on_click(save_and_extract)

display(ipyw.widgets.HBox([thresh_wig, save_extract_btn]))
display(out_write_thresh)

Output()

## Step 7. Visualise extracted timeseries
Click the button below to plot the signals that were extracted from the previous step. You can then return to Step 3 if you wish to work on a new organoid contraction video :)


In [20]:
out_plots = ipyw.widgets.Output()

@out_plots.capture(clear_output=True)
def plot_res(b):
    plot_btn.disabled = True
    if subsample_regions==10:
      cols=2
      fig, ax = plt.subplots(5, cols, figsize=(8,16), sharex=True, gridspec_kw={'hspace': 0.5})
      fig.suptitle('Shifts of border areas', fontsize=16)
      for i in range(subsample_regions):
        ax[i//2, i%2].plot(np.arange(distances.shape[1]), distances[i,:])
        ax[i//2, i%2].set_title('Area '+str(i+1), fontsize=12)
        ax[i//2, i%2].set_xlabel('frame id', fontsize=10)
        ax[i//2, i%2].set_ylabel('Area shift [pix]', fontsize=10)
    else:
      fig, ax = plt.subplots(5, figsize=(8,16), sharex=True, gridspec_kw={'hspace': 0.5})
      fig.suptitle('Shifts of border areas', fontsize=16)
      for i in range(subsample_regions):
        ax[i].plot(np.arange(distances.shape[1]), distances[i,:])
        ax[i].set_title('Area '+str(i+1), fontsize=12)
        ax[i].set_xlabel('frame id', fontsize=10)
        ax[i].set_ylabel('Area shift [pix]', fontsize=10)  
    plt.show()
    plot_btn.disabled = False
    
plot_btn = ipyw.widgets.Button(
    description='Plot results',
    disabled=False, # True
    button_style='',
    layout=ipyw.widgets.Layout(width='30%')
)
plot_btn.on_click(plot_res)


display(plot_btn)
display(out_plots)

Button(description='Plot results', layout=Layout(width='30%'), style=ButtonStyle())

Output()